# Week 1.2: How a Computer Turns Words into Numbers
### Plain-English edition — no coding required

A computer cannot read. It can only count. So before an AI can answer a question,
every word has to become a number.

This notebook shows you exactly how that happens, in three moves:
**chop the text up → give every piece a number → arrange the numbers so similar words sit together.**

## How this notebook works
Every grey box below is a **step you run**, not code you read.

1. Some steps have **boxes to fill in** (type a word, drag a slider, tick a box).
2. Press the **▶ play button** on the left of the step.
3. Read the answer that appears underneath.

That's it. The computer writes and runs the code for you.

**Curious what it wrote?** Click **Show code** on any step to read it, then click
**Hide code** in the same place to tuck it away again.

> **Red text means something needs fixing** — read it and press play again.
> **Green means you got it right.** No red at all? You're good.

## What you will learn
- **Tokenization** — chopping text into pieces a computer can handle
- **Subwords** — how a model copes with a word it has never seen
- **Embeddings** — turning each piece into a list of numbers that carries meaning
- **The map** — why "food" ends up near "rice" and far from "laptop"

---
## Part 1: Tokenization — chopping text into pieces

An AI never sees your sentence as a sentence. It sees a list of small pieces called **tokens**.
Type anything below and watch it happen.

In [ ]:
#@title ▶ Step 1 — Chop my sentence into pieces { display-mode: "form" }
#@markdown Type any sentence — about yourself, your day, anything.
my_sentence = "I love running in the morning and I eat rice for lunch" #@param {type:"string"}

# ---- hidden helper code: learners never need to read this ----
import sys, numpy as np, matplotlib.pyplot as plt
from collections import Counter
from IPython.display import display, HTML
def _msg(text, colour, weight):
    sys.stdout.flush()
    body = text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    display(HTML('<pre style="margin:0 0 2px 0;font-family:monospace;white-space:pre-wrap;'
                 'font-size:13px;color:' + colour + ';font-weight:' + weight + '">' + body + '</pre>'))
def problem(text): _msg(text, "#C8283C", "700")
def success(text): _msg(text, "#1a7f37", "700")
def note(text):    _msg(text, "#5f6368", "400")
def words_of(text):
    import re
    return [w for w in re.findall(r"[a-zA-Z']+", text.lower()) if w]

text = my_sentence.strip()
if len(text) < 3:
    problem("Please type a sentence first, then press play again.")
else:
    toks = words_of(text)
    print("Your sentence: " + text)
    print("")
    print("Chopped into " + str(len(toks)) + " tokens:")
    print("")
    line = ""
    for t in toks:
        piece = "[" + t + "] "
        if len(line) + len(piece) > 70: print("   " + line); line = ""
        line += piece
    if line: print("   " + line)
    print("")
    print("Now every token gets a number, which is all the computer really keeps:")
    print("")
    vocab = sorted(set(toks))
    ids = {w: i + 1 for i, w in enumerate(vocab)}
    print("   " + ", ".join(str(ids[t]) for t in toks))
    print("")
    print("Different words: " + str(len(vocab)) + " | Total tokens: " + str(len(toks)))
    rep = [w for w, c in Counter(toks).items() if c > 1]
    if rep:
        note("Repeated words get the SAME number every time: " + ", ".join(rep[:5]))
    print("")
    print("So what? Your sentence is now just a list of numbers.")
    print("That list is all an LLM ever receives - it never sees letters at all.")

### 🎯 Exercise 1.1: What happens to a word the model has never met?

Real models do not store every word. They keep a dictionary of about 100,000 pieces,
and build unknown words out of smaller chunks they already know.

Try a long or unusual word below — a name, a place, a technical term.

In [ ]:
#@title ▶ Step 2 — Break a rare word into familiar chunks { display-mode: "form" }
#@markdown Try something unusual: your surname, a town, a long word like "unbelievable".
rare_word = "pretraining" #@param {type:"string"}

# ---- hidden helper code: learners never need to read this ----
import sys, numpy as np, matplotlib.pyplot as plt
from collections import Counter
from IPython.display import display, HTML
def _msg(text, colour, weight):
    sys.stdout.flush()
    body = text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    display(HTML('<pre style="margin:0 0 2px 0;font-family:monospace;white-space:pre-wrap;'
                 'font-size:13px;color:' + colour + ';font-weight:' + weight + '">' + body + '</pre>'))
def problem(text): _msg(text, "#C8283C", "700")
def success(text): _msg(text, "#1a7f37", "700")
def note(text):    _msg(text, "#5f6368", "400")
def words_of(text):
    import re
    return [w for w in re.findall(r"[a-zA-Z']+", text.lower()) if w]

KNOWN = ["pre", "un", "re", "in", "dis", "over", "under", "train", "believ", "read", "play",
         "work", "run", "walk", "talk", "ing", "ed", "er", "ers", "s", "able", "ible", "ly",
         "ness", "ment", "tion", "sion", "est", "ful", "less", "ity", "ous", "al", "ive"]
w = rare_word.strip().lower()
if not w.isalpha():
    problem("Please type a single word, letters only.")
else:
    pieces, i = [], 0
    while i < len(w):
        hit = ""
        for k in sorted(KNOWN, key=len, reverse=True):
            if w.startswith(k, i) and len(k) > len(hit): hit = k
        if hit: pieces.append(hit); i += len(hit)
        else:
            if pieces and pieces[-1].startswith("?"): pieces[-1] += w[i]
            else: pieces.append("?" + w[i])
            i += 1
    shown = [p[1:] if p.startswith("?") else p for p in pieces]
    print("Your word: " + w)
    print("")
    print("Broken into pieces the model already knows:")
    print("")
    print("   " + "  +  ".join(shown))
    print("")
    known = [p for p in pieces if not p.startswith("?")]
    print(str(len(known)) + " of " + str(len(pieces)) + " pieces were already in the dictionary.")
    print("")
    if len(pieces) == 1:
        print("That whole word was already known - a very common word.")
    else:
        print("So what? The model has probably never seen '" + w + "' before.")
        print("It does not panic. It spells the word out of familiar parts,")
        print("the way you would sound out an unfamiliar name.")
    note("This is why an AI can handle your name, your town, or a word invented yesterday.")

---
## Part 2: Embeddings — numbers that carry meaning

A number like `17` tells the computer nothing about what a word *means*.

So each word gets a whole **list** of numbers instead, arranged so that words used in
similar ways end up with similar lists. We trained a small one on a handful of sentences
about exercise, food, computers, family and school.

In [ ]:
#@title ▶ Step 3 — Show me the numbers behind a word { display-mode: "form" }
#@markdown Pick a word to look inside. Try: running, rice, python, family, school.
word = "running" #@param {type:"string"}

# ---- hidden helper code: learners never need to read this ----
import sys, numpy as np, matplotlib.pyplot as plt
from collections import Counter
from IPython.display import display, HTML
def _msg(text, colour, weight):
    sys.stdout.flush()
    body = text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    display(HTML('<pre style="margin:0 0 2px 0;font-family:monospace;white-space:pre-wrap;'
                 'font-size:13px;color:' + colour + ';font-weight:' + weight + '">' + body + '</pre>'))
def problem(text): _msg(text, "#C8283C", "700")
def success(text): _msg(text, "#1a7f37", "700")
def note(text):    _msg(text, "#5f6368", "400")
def words_of(text):
    import re
    return [w for w in re.findall(r"[a-zA-Z']+", text.lower()) if w]

CORPUS = [
 "love running morning exercise keeps healthy fit",
 "running football swimming exercise sport enjoy",
 "exercise sport keeps body healthy strong fit",
 "football match team play sport running",
 "swimming running exercise healthy body",
 "eat rice beans lunch food tasty",
 "food rice beans chicken tasty filling lunch",
 "cooking food eating lunch family tasty",
 "rice chicken food dinner eating tasty",
 "write code python computer program software",
 "python code laptop computer software program",
 "programming code software laptop computer write",
 "computer laptop software program code",
 "family friends happy love together joy",
 "happy friends family laughter joy together",
 "sad tired alone unhappy feelings",
 "friends family together love happy",
 "school study books learning exams",
 "study books school learn exams reading",
 "school exams study learning books",
]
STOP = {"the","a","an","and","my","i","is","are","was","in","for","with","of","to","me","it","on","that","this"}
def content_words(text):
    return [w for w in words_of(text) if w not in STOP]
def build_space(dim=24, window=3):
    T = [content_words(c) for c in CORPUS]
    vocab = sorted({w for t in T for w in t}); idx = {w: i for i, w in enumerate(vocab)}
    n = len(vocab); M = np.zeros((n, n))
    for t in T:
        for i, w in enumerate(t):
            for j in range(max(0, i - window), min(len(t), i + window + 1)):
                if i != j: M[idx[w], idx[t[j]]] += 1.0
    tot = M.sum(); rows = M.sum(1, keepdims=True); cols = M.sum(0, keepdims=True)
    with np.errstate(divide="ignore", invalid="ignore"):
        P = np.log((M * tot) / (rows * cols + 1e-12) + 1e-12)
    P[~np.isfinite(P)] = 0; P[P < 0] = 0
    U, S, Vt = np.linalg.svd(P, full_matrices=False)
    V = U[:, :dim] * S[:dim]
    V = V / (np.linalg.norm(V, axis=1, keepdims=True) + 1e-9)
    return vocab, idx, V
def space():
    if "_SPACE" not in globals(): globals()["_SPACE"] = build_space()
    return globals()["_SPACE"]
def near(word, k=5):
    vocab, idx, V = space()
    if word not in idx: return None
    sims = V.dot(V[idx[word]])
    order = np.argsort(-sims)
    return [(vocab[i], float(sims[i])) for i in order if vocab[i] != word][:k]

vocab, idx, V = space()
w = word.strip().lower()
if w not in idx:
    problem("The word '" + w + "' is not in this small practice dictionary.")
    note("Try one of these: " + ", ".join(vocab[:18]) + " ...")
else:
    v = V[idx[w]]
    print("The word '" + w + "' is stored as this list of numbers:")
    print("")
    for r in range(0, 24, 8):
        print("   " + "  ".join(format(x, ">6.2f") for x in v[r:r + 8]))
    print("")
    print("That is 24 numbers. A real model like GPT uses a few thousand per word.")
    print("")
    print("On their own these numbers mean nothing. What matters is that words")
    print("used in similar ways end up with similar lists - which is Step 4.")
    note("Dictionary size here: " + str(len(vocab)) + " words. A real one holds about 100,000 pieces.")

In [ ]:
#@title ▶ Step 4 — Which words does the computer think are similar? { display-mode: "form" }
#@markdown Pick a word and see its nearest neighbours in meaning.
word = "running" #@param {type:"string"}
how_many = 5 #@param {type:"slider", min:3, max:10, step:1}

# ---- hidden helper code: learners never need to read this ----
import sys, numpy as np, matplotlib.pyplot as plt
from collections import Counter
from IPython.display import display, HTML
def _msg(text, colour, weight):
    sys.stdout.flush()
    body = text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    display(HTML('<pre style="margin:0 0 2px 0;font-family:monospace;white-space:pre-wrap;'
                 'font-size:13px;color:' + colour + ';font-weight:' + weight + '">' + body + '</pre>'))
def problem(text): _msg(text, "#C8283C", "700")
def success(text): _msg(text, "#1a7f37", "700")
def note(text):    _msg(text, "#5f6368", "400")
def words_of(text):
    import re
    return [w for w in re.findall(r"[a-zA-Z']+", text.lower()) if w]

CORPUS = [
 "love running morning exercise keeps healthy fit",
 "running football swimming exercise sport enjoy",
 "exercise sport keeps body healthy strong fit",
 "football match team play sport running",
 "swimming running exercise healthy body",
 "eat rice beans lunch food tasty",
 "food rice beans chicken tasty filling lunch",
 "cooking food eating lunch family tasty",
 "rice chicken food dinner eating tasty",
 "write code python computer program software",
 "python code laptop computer software program",
 "programming code software laptop computer write",
 "computer laptop software program code",
 "family friends happy love together joy",
 "happy friends family laughter joy together",
 "sad tired alone unhappy feelings",
 "friends family together love happy",
 "school study books learning exams",
 "study books school learn exams reading",
 "school exams study learning books",
]
STOP = {"the","a","an","and","my","i","is","are","was","in","for","with","of","to","me","it","on","that","this"}
def content_words(text):
    return [w for w in words_of(text) if w not in STOP]
def build_space(dim=24, window=3):
    T = [content_words(c) for c in CORPUS]
    vocab = sorted({w for t in T for w in t}); idx = {w: i for i, w in enumerate(vocab)}
    n = len(vocab); M = np.zeros((n, n))
    for t in T:
        for i, w in enumerate(t):
            for j in range(max(0, i - window), min(len(t), i + window + 1)):
                if i != j: M[idx[w], idx[t[j]]] += 1.0
    tot = M.sum(); rows = M.sum(1, keepdims=True); cols = M.sum(0, keepdims=True)
    with np.errstate(divide="ignore", invalid="ignore"):
        P = np.log((M * tot) / (rows * cols + 1e-12) + 1e-12)
    P[~np.isfinite(P)] = 0; P[P < 0] = 0
    U, S, Vt = np.linalg.svd(P, full_matrices=False)
    V = U[:, :dim] * S[:dim]
    V = V / (np.linalg.norm(V, axis=1, keepdims=True) + 1e-9)
    return vocab, idx, V
def space():
    if "_SPACE" not in globals(): globals()["_SPACE"] = build_space()
    return globals()["_SPACE"]
def near(word, k=5):
    vocab, idx, V = space()
    if word not in idx: return None
    sims = V.dot(V[idx[word]])
    order = np.argsort(-sims)
    return [(vocab[i], float(sims[i])) for i in order if vocab[i] != word][:k]

vocab, idx, V = space()
w = word.strip().lower()
if w not in idx:
    problem("The word '" + w + "' is not in this small practice dictionary.")
    note("Try one of these: " + ", ".join(vocab[:18]) + " ...")
else:
    res = near(w, int(how_many))
    print("Words the computer thinks are closest to '" + w + "':")
    print("")
    for other, sim in res:
        bar = "#" * max(0, int(round(sim * 25)))
        print("   " + other.ljust(14) + format(sim, ">6.2f") + "  " + bar)
    print("")
    print("The score runs from -1 (opposite) to 1 (identical).")
    print("")
    print("So what? Nobody told the computer what these words mean.")
    print("It noticed which words turn up in similar company, and that was enough.")
    note("That is the whole trick behind an LLM 'understanding' language.")

In [ ]:
#@title ▶ Step 5 — Draw me the map of word-space { display-mode: "form" }
#@markdown Words close together on this map are used in similar ways.
highlight_word = "rice" #@param {type:"string"}

# ---- hidden helper code: learners never need to read this ----
import sys, numpy as np, matplotlib.pyplot as plt
from collections import Counter
from IPython.display import display, HTML
def _msg(text, colour, weight):
    sys.stdout.flush()
    body = text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    display(HTML('<pre style="margin:0 0 2px 0;font-family:monospace;white-space:pre-wrap;'
                 'font-size:13px;color:' + colour + ';font-weight:' + weight + '">' + body + '</pre>'))
def problem(text): _msg(text, "#C8283C", "700")
def success(text): _msg(text, "#1a7f37", "700")
def note(text):    _msg(text, "#5f6368", "400")
def words_of(text):
    import re
    return [w for w in re.findall(r"[a-zA-Z']+", text.lower()) if w]

CORPUS = [
 "love running morning exercise keeps healthy fit",
 "running football swimming exercise sport enjoy",
 "exercise sport keeps body healthy strong fit",
 "football match team play sport running",
 "swimming running exercise healthy body",
 "eat rice beans lunch food tasty",
 "food rice beans chicken tasty filling lunch",
 "cooking food eating lunch family tasty",
 "rice chicken food dinner eating tasty",
 "write code python computer program software",
 "python code laptop computer software program",
 "programming code software laptop computer write",
 "computer laptop software program code",
 "family friends happy love together joy",
 "happy friends family laughter joy together",
 "sad tired alone unhappy feelings",
 "friends family together love happy",
 "school study books learning exams",
 "study books school learn exams reading",
 "school exams study learning books",
]
STOP = {"the","a","an","and","my","i","is","are","was","in","for","with","of","to","me","it","on","that","this"}
def content_words(text):
    return [w for w in words_of(text) if w not in STOP]
def build_space(dim=24, window=3):
    T = [content_words(c) for c in CORPUS]
    vocab = sorted({w for t in T for w in t}); idx = {w: i for i, w in enumerate(vocab)}
    n = len(vocab); M = np.zeros((n, n))
    for t in T:
        for i, w in enumerate(t):
            for j in range(max(0, i - window), min(len(t), i + window + 1)):
                if i != j: M[idx[w], idx[t[j]]] += 1.0
    tot = M.sum(); rows = M.sum(1, keepdims=True); cols = M.sum(0, keepdims=True)
    with np.errstate(divide="ignore", invalid="ignore"):
        P = np.log((M * tot) / (rows * cols + 1e-12) + 1e-12)
    P[~np.isfinite(P)] = 0; P[P < 0] = 0
    U, S, Vt = np.linalg.svd(P, full_matrices=False)
    V = U[:, :dim] * S[:dim]
    V = V / (np.linalg.norm(V, axis=1, keepdims=True) + 1e-9)
    return vocab, idx, V
def space():
    if "_SPACE" not in globals(): globals()["_SPACE"] = build_space()
    return globals()["_SPACE"]
def near(word, k=5):
    vocab, idx, V = space()
    if word not in idx: return None
    sims = V.dot(V[idx[word]])
    order = np.argsort(-sims)
    return [(vocab[i], float(sims[i])) for i in order if vocab[i] != word][:k]

vocab, idx, V = space()
PICK = ["running", "exercise", "football", "swimming", "healthy",
        "rice", "beans", "food", "lunch", "tasty",
        "python", "code", "computer", "laptop", "software",
        "family", "friends", "happy", "joy",
        "school", "study", "books", "exams"]
pts = [w for w in PICK if w in idx]
M = V[[idx[w] for w in pts]]
M = M - M.mean(0)
U, S, Wt = np.linalg.svd(M, full_matrices=False)
XY = M.dot(Wt[:2].T)
hl = highlight_word.strip().lower()
plt.figure(figsize=(10, 7))
groups = {"exercise": (0, 5, "seagreen"), "food": (5, 10, "darkorange"),
          "computers": (10, 15, "steelblue"), "people": (15, 19, "mediumpurple"),
          "school": (19, 23, "crimson")}
for name, (a, b, col) in groups.items():
    sel = [i for i in range(len(pts)) if a <= i < b]
    plt.scatter(XY[sel, 0], XY[sel, 1], s=90, color=col, label=name, alpha=0.85)
for i, w in enumerate(pts):
    plt.annotate(w, (XY[i, 0], XY[i, 1]), fontsize=9, xytext=(4, 4), textcoords="offset points")
if hl in pts:
    i = pts.index(hl)
    plt.scatter([XY[i, 0]], [XY[i, 1]], s=320, facecolors="none", edgecolors="black", linewidths=2)
plt.legend(loc="best"); plt.title("The map of meaning (24 numbers squashed down to 2)")
plt.xticks([]); plt.yticks([]); plt.tight_layout(); plt.show()
if hl in pts:
    print("'" + hl + "' is circled. Look at who its neighbours are.")
else:
    note("'" + hl + "' is not on this map. Try one of: " + ", ".join(pts[:10]) + " ...")
print("")
print("So what? Meaning became geography. Words about food cluster together,")
print("words about computers cluster somewhere else.")
print("Asking 'what is similar to rice?' is now just measuring a distance.")

---
## Part 3: The whole pipeline

Put the pieces together and you have the front half of every language model.

In [ ]:
#@title ▶ Step 6 — Run my sentence through the whole machine { display-mode: "form" }
#@markdown Watch your words become tokens, then numbers, then meaning.
my_sentence = "i love running and tasty food" #@param {type:"string"}

# ---- hidden helper code: learners never need to read this ----
import sys, numpy as np, matplotlib.pyplot as plt
from collections import Counter
from IPython.display import display, HTML
def _msg(text, colour, weight):
    sys.stdout.flush()
    body = text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    display(HTML('<pre style="margin:0 0 2px 0;font-family:monospace;white-space:pre-wrap;'
                 'font-size:13px;color:' + colour + ';font-weight:' + weight + '">' + body + '</pre>'))
def problem(text): _msg(text, "#C8283C", "700")
def success(text): _msg(text, "#1a7f37", "700")
def note(text):    _msg(text, "#5f6368", "400")
def words_of(text):
    import re
    return [w for w in re.findall(r"[a-zA-Z']+", text.lower()) if w]

CORPUS = [
 "love running morning exercise keeps healthy fit",
 "running football swimming exercise sport enjoy",
 "exercise sport keeps body healthy strong fit",
 "football match team play sport running",
 "swimming running exercise healthy body",
 "eat rice beans lunch food tasty",
 "food rice beans chicken tasty filling lunch",
 "cooking food eating lunch family tasty",
 "rice chicken food dinner eating tasty",
 "write code python computer program software",
 "python code laptop computer software program",
 "programming code software laptop computer write",
 "computer laptop software program code",
 "family friends happy love together joy",
 "happy friends family laughter joy together",
 "sad tired alone unhappy feelings",
 "friends family together love happy",
 "school study books learning exams",
 "study books school learn exams reading",
 "school exams study learning books",
]
STOP = {"the","a","an","and","my","i","is","are","was","in","for","with","of","to","me","it","on","that","this"}
def content_words(text):
    return [w for w in words_of(text) if w not in STOP]
def build_space(dim=24, window=3):
    T = [content_words(c) for c in CORPUS]
    vocab = sorted({w for t in T for w in t}); idx = {w: i for i, w in enumerate(vocab)}
    n = len(vocab); M = np.zeros((n, n))
    for t in T:
        for i, w in enumerate(t):
            for j in range(max(0, i - window), min(len(t), i + window + 1)):
                if i != j: M[idx[w], idx[t[j]]] += 1.0
    tot = M.sum(); rows = M.sum(1, keepdims=True); cols = M.sum(0, keepdims=True)
    with np.errstate(divide="ignore", invalid="ignore"):
        P = np.log((M * tot) / (rows * cols + 1e-12) + 1e-12)
    P[~np.isfinite(P)] = 0; P[P < 0] = 0
    U, S, Vt = np.linalg.svd(P, full_matrices=False)
    V = U[:, :dim] * S[:dim]
    V = V / (np.linalg.norm(V, axis=1, keepdims=True) + 1e-9)
    return vocab, idx, V
def space():
    if "_SPACE" not in globals(): globals()["_SPACE"] = build_space()
    return globals()["_SPACE"]
def near(word, k=5):
    vocab, idx, V = space()
    if word not in idx: return None
    sims = V.dot(V[idx[word]])
    order = np.argsort(-sims)
    return [(vocab[i], float(sims[i])) for i in order if vocab[i] != word][:k]

vocab, idx, V = space()
toks = words_of(my_sentence)
if not toks:
    problem("Please type a sentence first.")
else:
    print("STAGE 1 - your text")
    print("   " + my_sentence.strip())
    print("")
    print("STAGE 2 - chopped into tokens")
    print("   " + "  ".join("[" + t + "]" for t in toks))
    print("")
    print("STAGE 3 - each token becomes a number")
    ids = [idx.get(t, -1) for t in toks]
    print("   " + "  ".join((str(i) if i >= 0 else "?") for i in ids))
    print("")
    print("STAGE 4 - each number becomes a list of numbers carrying meaning")
    known = [t for t in toks if t in idx]
    if known:
        v = V[idx[known[0]]]
        print("   '" + known[0] + "' -> [" + ", ".join(format(x, ".2f") for x in v[:6]) + ", ...]")
    print("")
    print("STAGE 5 - the model adds up all that meaning and predicts what comes next")
    unknown = [t for t in toks if t not in idx]
    if unknown:
        note("Not in this small dictionary: " + ", ".join(unknown[:6]) +
             " (a real model would split these into subword pieces, as in Step 2)")
    print("")
    print("So what? Those five stages are the front half of ChatGPT.")
    print("The back half just predicts the next token - which is Week 1.1 all over again.")

### 🎯 Exercise: your digital twin's profile

Your twin needs to understand *you*. Describe yourself below and watch the computer
work out what your description is mostly about — using only the maths from Step 4.

In [ ]:
#@title ▶ Step 7 — What is my description really about? { display-mode: "form" }
#@markdown Describe yourself in a sentence or two: what you like, what you do.
about_me = "I love running and football, and I eat rice with my family" #@param {type:"string"}

# ---- hidden helper code: learners never need to read this ----
import sys, numpy as np, matplotlib.pyplot as plt
from collections import Counter
from IPython.display import display, HTML
def _msg(text, colour, weight):
    sys.stdout.flush()
    body = text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    display(HTML('<pre style="margin:0 0 2px 0;font-family:monospace;white-space:pre-wrap;'
                 'font-size:13px;color:' + colour + ';font-weight:' + weight + '">' + body + '</pre>'))
def problem(text): _msg(text, "#C8283C", "700")
def success(text): _msg(text, "#1a7f37", "700")
def note(text):    _msg(text, "#5f6368", "400")
def words_of(text):
    import re
    return [w for w in re.findall(r"[a-zA-Z']+", text.lower()) if w]

CORPUS = [
 "love running morning exercise keeps healthy fit",
 "running football swimming exercise sport enjoy",
 "exercise sport keeps body healthy strong fit",
 "football match team play sport running",
 "swimming running exercise healthy body",
 "eat rice beans lunch food tasty",
 "food rice beans chicken tasty filling lunch",
 "cooking food eating lunch family tasty",
 "rice chicken food dinner eating tasty",
 "write code python computer program software",
 "python code laptop computer software program",
 "programming code software laptop computer write",
 "computer laptop software program code",
 "family friends happy love together joy",
 "happy friends family laughter joy together",
 "sad tired alone unhappy feelings",
 "friends family together love happy",
 "school study books learning exams",
 "study books school learn exams reading",
 "school exams study learning books",
]
STOP = {"the","a","an","and","my","i","is","are","was","in","for","with","of","to","me","it","on","that","this"}
def content_words(text):
    return [w for w in words_of(text) if w not in STOP]
def build_space(dim=24, window=3):
    T = [content_words(c) for c in CORPUS]
    vocab = sorted({w for t in T for w in t}); idx = {w: i for i, w in enumerate(vocab)}
    n = len(vocab); M = np.zeros((n, n))
    for t in T:
        for i, w in enumerate(t):
            for j in range(max(0, i - window), min(len(t), i + window + 1)):
                if i != j: M[idx[w], idx[t[j]]] += 1.0
    tot = M.sum(); rows = M.sum(1, keepdims=True); cols = M.sum(0, keepdims=True)
    with np.errstate(divide="ignore", invalid="ignore"):
        P = np.log((M * tot) / (rows * cols + 1e-12) + 1e-12)
    P[~np.isfinite(P)] = 0; P[P < 0] = 0
    U, S, Vt = np.linalg.svd(P, full_matrices=False)
    V = U[:, :dim] * S[:dim]
    V = V / (np.linalg.norm(V, axis=1, keepdims=True) + 1e-9)
    return vocab, idx, V
def space():
    if "_SPACE" not in globals(): globals()["_SPACE"] = build_space()
    return globals()["_SPACE"]
def near(word, k=5):
    vocab, idx, V = space()
    if word not in idx: return None
    sims = V.dot(V[idx[word]])
    order = np.argsort(-sims)
    return [(vocab[i], float(sims[i])) for i in order if vocab[i] != word][:k]

vocab, idx, V = space()
THEMES = {"exercise and sport": ["running", "exercise", "football", "swimming", "healthy", "sport", "body"],
          "food and eating":    ["rice", "beans", "food", "lunch", "tasty", "cooking", "eating", "chicken"],
          "computers and code": ["python", "code", "computer", "laptop", "software", "programming"],
          "people and feelings":["family", "friends", "happy", "love", "together", "joy"],
          "school and study":   ["school", "study", "books", "learning", "exams", "learn"]}
toks = [t for t in content_words(about_me) if t in idx]
if len(toks) < 2:
    problem("I could not find enough familiar words in that description.")
    note("Try using some of these: running, football, rice, food, family, python, school, books")
else:
    me = V[[idx[t] for t in toks]].mean(0)
    me = me / (np.linalg.norm(me) + 1e-9)
    scores = []
    for name, ws in THEMES.items():
        ws = [w for w in ws if w in idx]
        if ws:
            tv = V[[idx[w] for w in ws]].mean(0)
            tv = tv / (np.linalg.norm(tv) + 1e-9)
            scores.append((float(me.dot(tv)), name))
    scores.sort(reverse=True)
    print("Words I recognised in your description:")
    print("   " + ", ".join(toks))
    print("")
    print("What your description is mostly about:")
    print("")
    for sc, name in scores:
        bar = "#" * max(0, int(round((sc + 1) * 14)))
        print("   " + name.ljust(22) + format(sc, ">6.2f") + "  " + bar)
    print("")
    print("Top match: " + scores[0][1].upper())
    print("")
    print("So what? Your twin just read a sentence it had never seen")
    print("and worked out the topic - with no rules, only distances between numbers.")

---
## Summary & Key Takeaways

1. **Tokens** — text is chopped into pieces before anything else happens.
2. **Subwords** — unknown words are spelled out of familiar chunks, so nothing is off-limits.
3. **Embeddings** — each piece becomes a list of numbers, learned from the company it keeps.
4. **The map** — similar meanings end up near each other, so "similar" becomes "close".
5. **The pipeline** — text → tokens → numbers → meaning → prediction.

> You never wrote code. You still saw exactly how a sentence becomes mathematics.

### Next up
How a model decides which words in your sentence matter most — **attention**.